# Latent Granular Resynthesis Using Neural Audio Codecs — Music2Latent Backend

Implementation inspired from **"Latent Granular Resynthesis Using Neural Audio Codecs"** (Tokui & Baker, ISMIR 2025).

Implementation of the pipeline for audio resynthesis that operates
by reworking granular synthesis at the latent vector level of **Music2Latent** (Pasini et al., ISMIR 2024),
a Consistency Autoencoder that compresses audio into continuous 64-dimensional latent vectors at ~10 Hz frame rate.

## Pipeline Overview

The algorithm has three stages:

### Stage 1 — Codebook Generation
- Encode source audio corpus with Music2Latent
- Segment latent sequence into **grains** (mean-pooled groups of consecutive frames)
- Key parameters: `grain_size` (frames per grain), `stride` (step between grains)
- Optionally augment source with pitch shifts to expand timbral coverage

### Stage 2 — Target Matching
- Encode target audio with the same codec
- Segment into grains of the same `grain_size`
- Match each target grain to the closest codebook grain via cosine similarity
- Temperature τ controls selection: τ→0 = deterministic best match, τ high = diverse/random

$$P(\text{select } B_i) = \frac{\exp(-D_{\cos}(A, B_i) / \tau)}{\sum_j \exp(-D_{\cos}(A, B_j) / \tau)}$$

### Stage 3 — Reconstruction
- Concatenate matched grain sequence into a hybrid latent tensor
- Decode through the codec's decoder to produce audio
- The decoder's upsampling implicitly interpolates between grains

## Phase 1 — Setup and Model Loading

Install and import dependencies, load Music2Latent, and verify encoder output shape.

In [ ]:
# pip install music2latent torchaudio soundfile matplotlib torch
import torch
import torch.nn.functional as F
import torchaudio
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import Audio, display
from music2latent import EncoderDecoder

# ---- Constants ----
SR = 44100           # Music2Latent sample rate
LATENT_DIM = 64      # Latent vector dimension

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
encdec = EncoderDecoder(device=DEVICE)

print(f"Device: {DEVICE}")
print(f"Sample rate: {SR}")
print(f"Latent dim: {LATENT_DIM}")

# Derive exact frame rate from a 1-second test encode (~4096 samples/frame)
dummy = np.zeros(SR, dtype=np.float32)  # 1 second of silence
test_latent = encdec.encode(dummy)
test_latent_np = np.array(test_latent)
FRAME_RATE_HZ = float(test_latent_np.shape[-1])   # frames produced for exactly SR samples
print(f"\nLatent shape for 1s audio: {test_latent_np.shape}")
print(f"Exact frame rate: {FRAME_RATE_HZ:.4f} Hz")
print(f"Latent dim: {test_latent_np.shape[1]}")
del dummy, test_latent, test_latent_np

## Phase 2 — Core Utility Functions

Import utility functions from `m2l_utils.py` and define the codec-agnostic resynthesis function.

In [ ]:
from utils.m2l_utils import (
    load_wav_mono,
    encode_continuous,
    build_granular_codebook,
    augment_and_build_codebook,
    decode_latents,
)

### `latent_granular_resynthesis` — Full paper algorithm with temperature sampling

This function is codec-agnostic — it operates purely on `(T, D)` torch tensors and is
identical to the EnCodec v2 notebook version.

In [ ]:
def latent_granular_resynthesis(target_latents, pool_grains, pool_grain_frames,
                               grain_size, temperature=0.0, random_seed=None):
    """
    Latent granular resynthesis: match target grains to source pool grains.

    Implements the full algorithm from Tokui & Baker (2025):
    1. Segment target latents into non-overlapping grains (stride = grain_size)
    2. Compute cosine similarity between target and pool grains (mean-pooled)
    3. Select matches via argmax (tau=0) or softmax sampling (tau>0)
    4. Concatenate original frame sequences of matched grains
    5. Trim to original target length (removes zero-pad overshoot from step 1)

    Parameters
    ----------
    target_latents : torch.Tensor
        Target latent tensor of shape (T_target, D).
    pool_grains : torch.Tensor
        Mean-pooled pool codebook grains of shape (N_pool, D).
    pool_grain_frames : torch.Tensor
        Original frame sequences of pool grains, shape (N_pool, grain_size, D).
    grain_size : int
        Number of consecutive latent frames per grain.
    temperature : float
        Sampling temperature. 0.0 = deterministic argmax (tau -> 0).
        Higher values introduce randomness/diversity.
    random_seed : int or None
        Random seed for reproducibility when temperature > 0.

    Returns
    -------
    hybrid_latents : torch.Tensor
        Hybrid latent tensor of shape (T_target, D).
    match_indices : torch.Tensor
        Pool grain index selected for each target grain, shape (N_target,).
    match_similarities : torch.Tensor
        Cosine similarity of the selected pool grain for each target grain,
        shape (N_target_grains,). Values in [-1, 1]; higher = better match.
    """
    T_target = target_latents.shape[0]
    device = target_latents.device
    pool_grains = pool_grains.to(device)
    pool_grain_frames = pool_grain_frames.to(device)

    # 1. Build target grains (non-overlapping: stride = grain_size).
    #    Pad to the next multiple of grain_size so the tail is never discarded.
    D = target_latents.shape[1]
    remainder = T_target % grain_size
    if remainder != 0:
        pad_frames = grain_size - remainder
        padding = torch.zeros(pad_frames, D, device=device)
        target_latents_for_grains = torch.cat([target_latents, padding], dim=0)
    else:
        target_latents_for_grains = target_latents
    target_grains, _ = build_granular_codebook(target_latents_for_grains, grain_size, grain_size)

    # 2. Cosine similarity matrix: (N_target, N_pool)
    t_norm = F.normalize(target_grains, dim=-1)
    p_norm = F.normalize(pool_grains, dim=-1)
    sim = torch.mm(t_norm, p_norm.T)

    # 3. Selection: argmax (tau effectively 0) or softmax sampling
    if temperature < 1e-6:
        match_indices = sim.argmax(dim=-1)
    else:
        # D_cos = 1 - cosine_similarity  (cosine distance)
        d_cos = 1.0 - sim
        logits = -d_cos / temperature
        probs = F.softmax(logits, dim=-1)
        if random_seed is not None:
            torch.manual_seed(random_seed)
        match_indices = torch.multinomial(probs, num_samples=1).squeeze(-1)

    # Extract per-grain match similarity
    match_similarities = sim[torch.arange(len(match_indices)), match_indices]

    # 4. Concatenate original frame sequences of matched grains
    matched_grain_frames = pool_grain_frames[match_indices]      # (N_target, grain_size, D)
    hybrid_latents = matched_grain_frames.reshape(-1, D)         # (N_target * grain_size, D)

    # 5. Trim to original target length (remove zero-pad overshoot)
    hybrid_latents = hybrid_latents[:T_target]

    n_unique = match_indices.unique().numel()
    print(f"  Unique grains used: {n_unique} / {pool_grains.shape[0]} pool grains")

    return hybrid_latents, match_indices, match_similarities

## Phase 3 — Load Audio

Load source and target wav files (mono, 44100 Hz). Change the paths below to use your own audio.

In [ ]:
# ---- Set file paths here ----
SOURCE_PATH = "wav44k/kitchen_pan_mono_44k.wav"   # Source corpus (timbre donor)
TARGET_PATH = "wav44k/fatty_mono_44k.wav"          # Target signal (structure donor)

source_np = load_wav_mono(SOURCE_PATH, target_sr=SR)
target_np = load_wav_mono(TARGET_PATH, target_sr=SR)

print(f"Source: {SOURCE_PATH}  shape={source_np.shape}  "
      f"duration={len(source_np)/SR:.2f}s")
print(f"Target: {TARGET_PATH}  shape={target_np.shape}  "
      f"duration={len(target_np)/SR:.2f}s")

print("\nSource audio:")
display(Audio(source_np, rate=SR))
print("Target audio:")
display(Audio(target_np, rate=SR))

## Phase 4 — Build the Granular Codebook

### Grain size at Music2Latent's ~10 Hz frame rate

Because Music2Latent operates at ~10 Hz (one latent frame ≈ 100ms), grain sizes carry
different temporal meaning:

- `grain_size=1` ≈ **100ms** — suitable for percussive / transient content
- `grain_size=2` ≈ **200ms** — good general-purpose setting
- `grain_size=3` ≈ **300ms** — suitable for harmonic / tonal content
- `grain_size=5` ≈ **500ms** — suitable for sustained pads or long tones

### Stride and augmentation

- **Smaller stride** → greater coverage through overlapping segments
- **Larger stride** → forces more diversity between grains

Pitch-shift augmentation (±1, ±2 semitones) expands the codebook's timbral coverage, especially
valuable when working with limited source material.

In [ ]:
# ---- Codebook parameters ----
GRAIN_SIZE = 1         # 1 frame ≈ 100ms at ~10 Hz
STRIDE = 1             # Step between grain starts (1 = no overlap)
SEMITONE_SHIFTS = [-2, -1, 1, 2]  # Pitch shifts for augmentation

print(f"Building codebook: grain_size={GRAIN_SIZE}, stride={STRIDE}, "
      f"shifts={SEMITONE_SHIFTS}")
print()

pool_grains, pool_grain_frames = augment_and_build_codebook(
    source_np, encdec,
    grain_size=GRAIN_SIZE,
    stride=STRIDE,
    semitone_shifts=SEMITONE_SHIFTS,
    device=DEVICE,
)
print(f"\nFinal pool shape: {tuple(pool_grains.shape)}")
print(f"Pool grain frames shape: {tuple(pool_grain_frames.shape)}")


## Phase 5 — Encode Target

Encode the target audio and verify with a round-trip decode.

In [ ]:
target_latents = encode_continuous(target_np, encdec).to(DEVICE)
print(f"Target latent shape: {tuple(target_latents.shape)}")
print(f"Target latents device: {target_latents.device}")

# Round-trip sanity check: decode and play to verify reconstruction quality
target_roundtrip_np = decode_latents(target_latents, encdec)
print(f"Round-trip audio shape: {target_roundtrip_np.shape}")

print("\nTarget round-trip (should sound like the original):")
display(Audio(target_roundtrip_np, rate=SR))

## Phase 6 — Resynthesis

Run the latent granular resynthesis and compare source / target / hybrid.

In [ ]:
# ---- Resynthesis parameters ----
TEMPERATURE = 0.0   # 0.0 = deterministic argmax; increase for randomness
RANDOM_SEED = 42    # set to None to disable; only used when TEMPERATURE > 0

print(f"Resynthesis: grain_size={GRAIN_SIZE}, temperature={TEMPERATURE}")
print()

hybrid_latents, match_indices, match_similarities = latent_granular_resynthesis(
    target_latents, pool_grains, pool_grain_frames,
    grain_size=GRAIN_SIZE,
    temperature=TEMPERATURE,
    random_seed=RANDOM_SEED
)

hybrid_np = decode_latents(hybrid_latents, encdec)

# Source round-trip for comparison
source_latents = encode_continuous(source_np, encdec).to(DEVICE)
source_roundtrip_np = decode_latents(source_latents, encdec)

print("\n--- A/B/C Comparison ---")
print("A) Source round-trip (corpus / timbre donor):")
display(Audio(source_roundtrip_np, rate=SR))

print("\nB) Target round-trip (structure donor):")
display(Audio(target_roundtrip_np, rate=SR))

print("\nC) Hybrid (source timbre × target structure):")
display(Audio(hybrid_np, rate=SR))

OUTPUT_PATH = "hybrid_output.wav"
sf.write(OUTPUT_PATH, hybrid_np, SR)
print(f"Hybrid audio saved to: {OUTPUT_PATH}")

## Compute-Intensive Grid Search

The following cell runs resynthesis across multiple `grain_size` and `temperature` combinations.
**On CPU this can be slow** (several minutes). To speed it up:
- Reduce `GRAIN_SIZES` (e.g., `[1, 2]`)
- Reduce `TEMPERATURES` (e.g., `[0.0, 1.0]`)
- Use a GPU if available

## Phase 7 — Parameter Exploration

Grid search over grain sizes and temperatures to hear their effect.
Grain sizes adjusted for Music2Latent's ~10 Hz frame rate (1–3 instead of 1–4).

In [ ]:
# ---- Grid parameters ----
GRAIN_SIZES = [1, 2, 3]
TEMPERATURES = [0.0, 0.5, 1.5]
GRID_STRIDE = 1

# --- Encode once: original + all pitch-shifted variants ---
print("Pre-encoding source audio (original + pitch shifts) — runs once only...")
_latents_cache = {}

_latents_cache["original"] = encode_continuous(source_np, encdec)
print(f"  Original encoded: {tuple(_latents_cache['original'].shape)}")

for _shift in SEMITONE_SHIFTS:
    _audio_tensor = torch.from_numpy(source_np).unsqueeze(0)
    _pitch_shifter = torchaudio.transforms.PitchShift(SR, _shift)
    _shifted_np = _pitch_shifter(_audio_tensor).squeeze(0).detach().numpy().astype(np.float32)
    _latents_cache[_shift] = encode_continuous(_shifted_np, encdec)
    print(f"  Shift {_shift:+d} semitones encoded: {tuple(_latents_cache[_shift].shape)}")

print(f"\nEncoding complete. Running grid: {len(GRAIN_SIZES)} grain sizes × "
      f"{len(TEMPERATURES)} temperatures = {len(GRAIN_SIZES)*len(TEMPERATURES)} variants\n")

# --- Grid search: only re-pool for each grain_size, no re-encoding ---
for gs in GRAIN_SIZES:
    print(f"\n{'='*60}")
    print(f"Building pool for grain_size={gs} (pooling cached latents)")

    _grains_list = []
    _frames_list = []
    for _key in ["original"] + list(SEMITONE_SHIFTS):
        _g, _f = build_granular_codebook(_latents_cache[_key], gs, GRID_STRIDE)
        _grains_list.append(_g)
        _frames_list.append(_f)
    pool_gs = torch.cat(_grains_list, dim=0)
    pool_gs_frames = torch.cat(_frames_list, dim=0)
    print(f"  Pool shape: {tuple(pool_gs.shape)}")

    for temp in TEMPERATURES:
        print(f"\n--- grain_size={gs}, tau={temp} ---")
        hybrid_lat, _, _ = latent_granular_resynthesis(
            target_latents, pool_gs, pool_gs_frames,
            grain_size=gs,
            temperature=temp,
            random_seed=RANDOM_SEED
        )
        audio_out = decode_latents(hybrid_lat, encdec)
        label = f"grain_size={gs}_tau={temp}"
        out_path = f"hybrid_{label}.wav"
        sf.write(out_path, audio_out, SR)
        print(f"Saved: {out_path}")
        display(Audio(audio_out, rate=SR))

## Phase 8 — Visualization

Waveform comparison, per-frame cosine similarity, and grain usage histogram.

In [ ]:
# Waveform data
src_np = source_roundtrip_np.squeeze()
tgt_np = target_roundtrip_np.squeeze()
hyb_np = hybrid_np.squeeze()

# Per-grain match similarity (what the algorithm actually used)
T_target = target_latents.shape[0]
sim_np = match_similarities.cpu().numpy()          # (N_grains,)
# Expand each grain's score across its constituent frames for step-plot display
sim_per_frame = np.repeat(sim_np, GRAIN_SIZE)[:T_target]   # trim to exact target length
frame_times = np.arange(len(sim_per_frame)) * (1.0 / FRAME_RATE_HZ)

fig = plt.figure(figsize=(14, 13))
fig.suptitle("Latent Granular Resynthesis (Music2Latent) — Analysis",
             fontsize=14, fontweight="bold")
gs_plot = gridspec.GridSpec(5, 1, hspace=0.65)

# Waveforms (3 subplots)
for i, (wav, label, colour) in enumerate([
    (src_np, "Source (pool / corpus)", "tab:green"),
    (tgt_np, "Target (structure donor)", "tab:blue"),
    (hyb_np, "Hybrid (source timbre × target structure)", "tab:orange"),
]):
    ax = fig.add_subplot(gs_plot[i])
    t_axis = np.linspace(0, len(wav) / SR, len(wav))
    ax.plot(t_axis, wav, lw=0.4, color=colour, rasterized=True)
    ax.set_ylabel("amp", fontsize=8)
    ax.set_title(label, fontsize=9, loc="left")
    ax.set_xlim([0, t_axis[-1]])

# Per-grain match similarity (subplot 4)
ax4 = fig.add_subplot(gs_plot[3])
ax4.step(frame_times, sim_per_frame, lw=0.8, color="tab:red", where="post")
ax4.axhline(sim_np.mean(), ls="--", lw=0.8, color="grey",
            label=f"mean = {sim_np.mean():.3f}")
ax4.set_ylabel("cosine sim", fontsize=8)
ax4.set_title(
    "Per-grain match similarity (selected pool grain vs target grain, "
    f"grain_size={GRAIN_SIZE})",
    fontsize=9, loc="left"
)
ax4.set_ylim([-0.05, 1.05])
ax4.legend(fontsize=8)
ax4.set_xlim([0, frame_times[-1]])

# Histogram of match indices (subplot 5) — diversity measure
ax5 = fig.add_subplot(gs_plot[4])
idx_np = match_indices.cpu().numpy()
n_bins = min(50, pool_grains.shape[0])
ax5.hist(idx_np, bins=n_bins, color="tab:purple", edgecolor="black", alpha=0.7)
ax5.set_xlabel("Pool grain index", fontsize=9)
ax5.set_ylabel("Count", fontsize=8)
n_unique = len(np.unique(idx_np))
ax5.set_title(f"Match index distribution — {n_unique} unique grains used "
              f"out of {pool_grains.shape[0]} (diversity measure)",
              fontsize=9, loc="left")

plt.savefig("latent_granular_resynthesis_music2latent.png", dpi=150, bbox_inches="tight")
plt.show()

## Phase 9 — Extended Methods: Viterbi Path Selection & Soft Blending

Implements the research extensions from *"Temporal Coherence via Viterbi Path Selection
in Latent Granular Resynthesis"*:

- **Viterbi selection** (Direction A): globally optimal grain sequence with smoothness penalty λ
- **Soft blending** (Direction B): weighted convex combination of top-k pool grains
- **Quantitative evaluation** (Direction C): MFCC timbre similarity + RMS envelope correlation

In [ ]:
# Phase 9 — Extended methods: imports & module reloads
import sys, importlib
sys.path.insert(0, ".")

import utils.m2l_utils, utils.viterbi_lgr, utils.soft_lgr, utils.evaluation
importlib.reload(utils.m2l_utils)
importlib.reload(utils.viterbi_lgr)
importlib.reload(utils.soft_lgr)
importlib.reload(utils.evaluation)

from utils.m2l_utils   import augment_and_build_codebook
from utils.viterbi_lgr import viterbi_granular_resynthesis, batch_viterbi_sweep
from utils.soft_lgr    import soft_granular_resynthesis
from utils.evaluation  import evaluate_all, compare_methods


## Phase 10 — Viterbi Resynthesis

Sweep over smoothness values λ ∈ {0, 0.5, 1, 2}.
λ=0 reproduces the baseline deterministic argmax exactly.

In [ ]:
# Phase 10 — Viterbi grain path resynthesis
# Uses pool_grains / pool_grain_frames / target_latents from Phase 4-5.
# Reuses GRAIN_SIZE from Phase 4.

SMOOTHNESS_VALUES = [0.0, 0.5, 1.0, 2.0]

print("Running Viterbi sweep...")
viterbi_results_raw = batch_viterbi_sweep(
    target_latents, pool_grains, pool_grain_frames,
    grain_size=GRAIN_SIZE,
    smoothness_values=SMOOTHNESS_VALUES,
)

# Decode all Viterbi outputs
viterbi_audio = {}
for r in viterbi_results_raw:
    lam = r["smoothness"]
    audio_out = decode_latents(r["hybrid_latents"], encdec)
    viterbi_audio[lam] = audio_out
    label = f"Viterbi λ={lam}"
    out_path = f"hybrid_viterbi_lambda{lam}.wav"
    sf.write(out_path, audio_out, SR)
    print(f"\n{label}:")
    display(Audio(audio_out, rate=SR))

## Phase 11 — Soft Latent Blending Ablation

Soft blending replaces hard grain selection with a weighted sum of the top-k
closest pool grains. Top-k=1 with tau→0 reduces to the hard argmax baseline.

In [ ]:
# Phase 11 — Soft blending ablation
TOP_K_VALUES  = [1, 3, 5, 10]
SOFT_TEMP     = 1.0

soft_audio = {}
for k in TOP_K_VALUES:
    hybrid_soft = soft_granular_resynthesis(
        target_latents, pool_grains, pool_grain_frames,
        grain_size=GRAIN_SIZE,
        top_k=k,
        temperature=SOFT_TEMP,
    )
    audio_out = decode_latents(hybrid_soft, encdec)
    soft_audio[k] = audio_out
    out_path = f"hybrid_soft_k{k}.wav"
    sf.write(out_path, audio_out, SR)
    label = f"Soft blend k={k}"
    print(f"\n{label}:")
    display(Audio(audio_out, rate=SR))

## Phase 12 — Quantitative Evaluation

Two metrics per method:
- **Timbre similarity** (MFCC cosine sim, ↑ = more source-like)
- **Structural preservation** (RMS envelope Pearson r, ↑ = follows target better)

In [ ]:
# Phase 12 — Quantitative evaluation
# Gather all methods into a single comparison.

results = []

# 1. Baseline: deterministic argmax (temperature=0)
hyb_base_0, _, _ = latent_granular_resynthesis(
    target_latents, pool_grains, pool_grain_frames,
    grain_size=GRAIN_SIZE, temperature=0.0,
)
results.append(evaluate_all(target_np, decode_latents(hyb_base_0, encdec),
                             SR, label="Baseline argmax"))

# 2. Baseline: softmax sampling (temperature=0.5)
hyb_base_05, _, _ = latent_granular_resynthesis(
    target_latents, pool_grains, pool_grain_frames,
    grain_size=GRAIN_SIZE, temperature=0.5, random_seed=42,
)
results.append(evaluate_all(target_np, decode_latents(hyb_base_05, encdec),
                             SR, label="Baseline tau=0.5"))

# 3. Soft blend variants
for k in TOP_K_VALUES:
    results.append(evaluate_all(target_np, soft_audio[k],
                                SR, label=f"Soft blend k={k}"))

# 4. Viterbi variants
for lam in SMOOTHNESS_VALUES:
    results.append(evaluate_all(target_np, viterbi_audio[lam],
                                SR, label=f"Viterbi lambda={lam}"))

print("\n" + "="*60)
compare_methods(results)

## Phase 13 — Latent Space Visualisation

UMAP (or PCA fallback) 2D projection of all pool codebook grains.

**Reading the plot:**
- Each point = one codebook grain (mean-pooled latent vector)
- **Colour** = source audio variant (original vs pitch-shifted)
- **Marker size** = how often the Viterbi algorithm selected that grain
  (large dot = frequently used; small dot = rarely or never selected)

Clusters of same-colour points confirm that pitch-shifted grains occupy distinct
but adjacent regions — evidence that Music2Latent's latent space encodes pitch
in a geometrically structured way.

In [ ]:
# Phase 13 — Latent space visualisation
import matplotlib.pyplot as plt
import numpy as np

try:
    import umap
    reducer_name = "UMAP"
    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
except ImportError:
    from sklearn.decomposition import PCA
    reducer_name = "PCA"
    reducer = PCA(n_components=2)

grains_np = pool_grains.cpu().numpy()
embedding = reducer.fit_transform(grains_np)

# Derive per-variant grain counts — PitchShift preserves length so all
# variants produce the same number of grains. No m2l_utils.py change needed.
n_variants = 1 + len(SEMITONE_SHIFTS)
n_per_variant = pool_grains.shape[0] // n_variants
pool_grain_counts = [n_per_variant] * n_variants

n_total = grains_np.shape[0]
colour_ids = np.concatenate([
    np.full(count, variant_idx, dtype=int)
    for variant_idx, count in enumerate(pool_grain_counts)
])
# Handle any remainder from integer division (last variant absorbs leftover grains)
if colour_ids.shape[0] < n_total:
    colour_ids = np.concatenate([
        colour_ids,
        np.full(n_total - colour_ids.shape[0], len(pool_grain_counts) - 1, dtype=int)
    ])
assert colour_ids.shape[0] == n_total, (
    f"colour_ids length {colour_ids.shape[0]} != pool size {n_total}"
)
print(f"Pool: {n_total} grains total, {n_per_variant} per variant × {n_variants} variants")

# Selection frequency from the middle smoothness value
mid_idx = len(SMOOTHNESS_VALUES) // 2
mid_lam = SMOOTHNESS_VALUES[mid_idx]
best_path = viterbi_results_raw[mid_idx]["path"]
freq = np.bincount(best_path.cpu().numpy(), minlength=n_total).astype(float)
freq_norm = freq / freq.max() if freq.max() > 0 else freq
marker_sizes = 20 + 200 * freq_norm

palette = ["#1D9E75", "#E8593C", "#3B8BD4", "#EF9F27", "#9B59B6"]
shift_labels = ["original"] + [f"shift {s:+d} st" for s in SEMITONE_SHIFTS]

fig, ax = plt.subplots(figsize=(9, 7))
for cid, (label, color) in enumerate(zip(shift_labels, palette)):
    mask = colour_ids == cid
    if not mask.any():
        continue
    ax.scatter(
        embedding[mask, 0], embedding[mask, 1],
        s=marker_sizes[mask], c=color, alpha=0.75,
        edgecolors="white", linewidths=0.4, label=label,
    )

ax.set_title(f"{reducer_name} projection of codebook grains\n"
             f"(marker size = selection frequency by Viterbi \u03bb={mid_lam})", fontsize=11)
ax.set_xlabel(f"{reducer_name} dim 1", fontsize=9)
ax.set_ylabel(f"{reducer_name} dim 2", fontsize=9)
ax.legend(fontsize=8, title="Source variant", title_fontsize=8)
ax.grid(True, alpha=0.2)
fig.tight_layout()
fig.savefig("latent_space_umap.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: latent_space_umap.png")

In [ ]:
# Phase 14 — Results figure (structure + per-grain comparison only)
# One figure with 2 subplots:
#   (a) Structural preservation vs lambda (Viterbi sweep)
#   (b) Per-grain match similarity comparison: argmax vs best Viterbi

# Use middle lambda value for subplot (b) — same as Phase 13
mid_idx = len(SMOOTHNESS_VALUES) // 2
mid_lam = SMOOTHNESS_VALUES[mid_idx]
best_viterbi_sims = viterbi_results_raw[mid_idx]["match_similarities"].cpu().numpy()

# Baseline argmax match similarities (recompute from Phase 6 data)
_, _, base_sims = latent_granular_resynthesis(
    target_latents, pool_grains, pool_grain_frames,
    grain_size=GRAIN_SIZE, temperature=0.0,
)
base_sims_np = base_sims.cpu().numpy()

# Extract Viterbi-only rows from results
viterbi_rows = [r for r in results if r["label"].startswith("Viterbi")]
lambdas      = [float(r["label"].split("=")[1]) for r in viterbi_rows]
struct_vals  = [r["structural_preservation"] for r in viterbi_rows]

fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle("Latent Granular Resynthesis — Extended Method Comparison", fontsize=13, fontweight="bold")
fig.subplots_adjust(hspace=0.45, wspace=0.35)

# (a) Structure vs lambda
ax_a.plot(lambdas, struct_vals, "s-", color="#3B8BD4", lw=2)
ax_a.axhline(results[0]["structural_preservation"], ls="--", color="gray", lw=1, label="Baseline argmax")
ax_a.set_xlabel("Smoothness \u03bb", fontsize=9)
ax_a.set_ylabel("Structural preservation (RMS corr) \u2191", fontsize=9)
ax_a.set_title("(a) Structure preservation vs Viterbi \u03bb", fontsize=10, loc="left")
ax_a.legend(fontsize=8)
ax_a.grid(True, alpha=0.2)

# (b) Per-grain match similarity comparison
grain_indices = np.arange(len(base_sims_np))
ax_b.step(grain_indices, base_sims_np,      lw=1.0, color="#888780", label="Baseline argmax", where="post")
ax_b.step(grain_indices, best_viterbi_sims, lw=1.0, color="#E8593C", label=f"Viterbi \u03bb={mid_lam}",  where="post")
ax_b.axhline(base_sims_np.mean(),      ls="--", lw=0.8, color="#888780", alpha=0.6)
ax_b.axhline(best_viterbi_sims.mean(), ls="--", lw=0.8, color="#E8593C", alpha=0.6)
ax_b.set_xlabel("Grain index (time \u2192)", fontsize=9)
ax_b.set_ylabel("Cosine similarity", fontsize=9)
ax_b.set_title("(b) Per-grain match quality: baseline vs Viterbi", fontsize=10, loc="left")
ax_b.legend(fontsize=8)
ax_b.set_ylim([-0.05, 1.05])
ax_b.grid(True, alpha=0.2)

fig.savefig("extended_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: extended_results.png")